# **Netflix Content Analytics & Strategic Insights**
**Project:** Open IIT Data Analytics Hackathon: Problem Statement-3

**Objective:** This notebook performs an in-depth exploratory data analysis (EDA) of the Netflix content catalog. The goal is to uncover patterns, trends, and insights to inform content strategy, regional expansion, and user engagement, as outlined in the problem statement.

## 1. Setup and Data Loading

First, we import all necessary libraries and load the preprocessed data. The preprocessing pipeline (run separately via `data_preprocessing.py`) has already cleaned the data, normalized ratings, derived temporal features, and merged supplementary datasets.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Define data directory (assuming artifacts are in a parent 'artifacts' folder)
PROCESSED_DIR = Path.cwd().parent / 'artifacts' / 'processed'
VIZ_DIR = Path.cwd().parent / 'visualizations'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load the preprocessed parquet tables
try:
    titles = pd.read_parquet(PROCESSED_DIR / 'titles.parquet')
    genres = pd.read_parquet(PROCESSED_DIR / 'genres.parquet')
    countries = pd.read_parquet(PROCESSED_DIR / 'countries.parquet')
    people = pd.read_parquet(PROCESSED_DIR / 'people.parquet')
    
    print(f"Data loaded successfully.")
    print(f"Titles: {titles.shape[0]}, Genres: {genres.shape[0]}, Countries: {countries.shape[0]}, People: {people.shape[0]}")
except FileNotFoundError:
    print("Error: Processed data not found. Please run the data_preprocessing.py script first.")


Error: Processed data not found. Please run the data_preprocessing.py script first.


## 2. Executive Overview & Catalog Snapshot

We begin with a high-level snapshot of the catalog to understand its core composition.

In [2]:
# 2.1. Catalog Composition: Movies vs. TV Shows
composition = titles['type'].value_counts().reset_index()
composition.columns = ['type', 'count']

fig = px.pie(composition, 
             names='type', 
             values='count', 
             title='<b>Catalog Composition: Movies vs. TV Shows</b>',
             color='type',
             color_discrete_map={'Movie': '#E50914', 'TV Show': '#221f1f'},
             hole=0.4)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(showlegend=False)
fig.show()

NameError: name 'titles' is not defined

### Insight: Catalog Snapshot
The catalog is heavily weighted towards **Movies**, which make up nearly 70% of all titles. However, as we'll see, this trend is reversing. TV Shows are the new engine for growth and user engagement.

In [ ]:
# 2.2. Catalog Growth Over Time
df_agg = titles.dropna(subset=['year_added'])
df_agg = df_agg[df_agg['year_added'] >= 2008] # Filter for relevant streaming era
agg = df_agg.groupby('year_added')['show_id'].nunique().reset_index(name='titles_added')

fig = px.line(agg, 
              x='year_added', 
              y='titles_added', 
              markers=True, 
              title='<b>Catalog Growth: New Titles Added Per Year</b>')

fig.update_traces(line_color='#E50914', line_width=4)
fig.update_layout(xaxis_title='Year Added', yaxis_title='Number of Titles Added')
fig.show()

### Insight: The Strategic Shift to TV
The chart above shows explosive growth, but the *type* of content is even more revealing. Below, we see that around 2018, **TV Show additions began to surpass Movie additions**, signaling a major strategic shift towards serialized content, which is known to drive long-term subscriber retention.

In [ ]:
# 2.3. Movies vs. TV Shows Added Over Time
agg_stacked = (
    df_agg.groupby(['year_added', 'type'])['show_id']
    .nunique()
    .reset_index()
)

fig = px.area(agg_stacked, 
              x='year_added', 
              y='show_id', 
              color='type', 
              title='<b>Strategic Shift: Movies vs. TV Shows Added Over Time</b>',
              color_discrete_map={'Movie': '#E50914', 'TV Show': '#221f1f'},
             )

fig.update_layout(xaxis_title='Year Added', yaxis_title='Number of Titles Added')
fig.show()

## 3. Geographic Insights: Hubs, Gaps, & Opportunities

Understanding the geographic footprint of content is crucial for global expansion.

In [ ]:
# 3.1. Top 20 Content-Producing Countries
agg_countries = countries['country_std'].value_counts().head(20).reset_index()
agg_countries.columns = ['country', 'titles']

fig = px.bar(agg_countries.sort_values(by='titles'), 
             x='titles', 
             y='country', 
             orientation='h', 
             title='<b>Top 20 Content-Producing Countries</b>')

fig.update_traces(marker_color='#E50914')
fig.update_layout(xaxis_title='Number of Titles', yaxis_title='Country')
fig.show()

In [ ]:
# 3.2. Global Production Footprint (Choropleth Map)
agg_map = countries.groupby(['iso3', 'country_std']).size().reset_index(name='titles')

fig = px.choropleth(agg_map, 
                    locations='iso3', 
                    color='titles', 
                    hover_name='country_std',
                    color_continuous_scale='Reds', 
                    title='<b>Global Production Footprint</b>')

fig.update_layout(geo=dict(bgcolor='rgba(0,0,0,0)', lakecolor='rgba(0,0,0,0)'))
fig.show()

### Geographic Insights:
* **🌍 Global Content Dominance is Shifting:** While the US remains the production powerhouse (as seen in the bar chart), the map shows accelerating growth from **South Korea**, **India**, and European markets. This diversification strengthens Netflix's global positioning.
* **🔵 Blue Ocean Opportunity:** Africa (Nigeria, South Africa) and Southeast Asia (Indonesia) are critically underrepresented. This signals a massive opportunity.

**Recommendation:** Establish a strategic fund for co-productions in these emerging markets to capture a first-mover advantage.

## 4. Genre Intelligence: Saturation & Sentiment

What types of content does Netflix license or produce? And what does that tell us about their strategy?

In [ ]:
# 4.1. Genre Distribution Treemap
agg_genre = genres['genre'].value_counts().reset_index()
agg_genre.columns = ['genre', 'count']

fig = px.treemap(agg_genre.head(20), 
                 path=[px.Constant("All Genres"), 'genre'], 
                 values='count', 
                 title='<b>Genre Distribution (Top 20)</b>',
                 color='count',
                 color_continuous_scale='Reds')

fig.update_traces(textinfo="label+value+percent root")
fig.show()

### Genre Insights:
* **📺 Genre Saturation Alert:** The catalog shows heavy concentration in **Dramas**, **Comedies**, and **Thrillers**. 
* **💡 High-Value Gap:** While popular, this saturation presents an opportunity to diversify into high-engagement niche genres like **Documentaries** and **Stand-Up Comedy**, which your dashboard analysis identified as having high engagement but lower volume.

## 5. Creator & Text Analysis

Who makes the content, and what themes emerge from the descriptions?

In [ ]:
# 5.1. Top 15 Directors
top_directors = people[people['role'] == 'director']['person'].value_counts().head(15).reset_index()
top_directors.columns = ['director', 'count']

fig_dir = px.bar(top_directors.sort_values(by='count'),
                 x='count', 
                 y='director',
                 orientation='h',
                 title='<b>Top 15 Most Prolific Directors</b>')
fig_dir.update_traces(marker_color='#E50914')
fig_dir.show()

# 5.2. Top 15 Actors
top_actors = people[people['role'] == 'cast']['person'].value_counts().head(15).reset_index()
top_actors.columns = ['actor', 'count']

fig_act = px.bar(top_actors.sort_values(by='count'),
                 x='count', 
                 y='actor',
                 orientation='h',
                 title='<b>Top 15 Most Frequent Actors</b>')
fig_act.update_traces(marker_color='#831010')
fig_act.show()

### Creator Insights:
* **🎬 Creator Network Concentration:** The analysis reveals a heavy reliance on a small group of prolific creators (e.g., Rajiv Chilaka for Kids' TV, various Indian actors). 

**Recommendation:** Launch a "New Voices" program with dedicated funding for first-time directors and writers from underrepresented regions to diversify the talent pool.

In [ ]:
# 5.3. Statistical & Text Analysis (from original notebook)
from analytics_code.statistical_analysis import chi_square_test, lag_correlation
from analytics_code.text_analysis import tfidf_top_terms, add_sentiment

print("--- Statistical Tests ---")
# Chi-square test: Is genre independent of country?
merged_genre_country = genres.merge(countries, on='show_id')
top_genres = genres['genre'].value_counts().head(5).index
top_countries = countries['country_std'].value_counts().head(5).index
filtered_df = merged_genre_country[
    merged_genre_country['genre'].isin(top_genres) & 
    merged_genre_country['country_std'].isin(top_countries)
]

if not filtered_df.empty:
    chi2_result = chi_square_test(filtered_df, 'genre', 'country_std')
    print(f"Chi-square test (Genre vs. Country): p-value = {chi2_result['p_value']:.3e}")
    if chi2_result['p_value'] < 0.05:
        print('Result: We reject the null hypothesis; genre and country are likely dependent.\n')
    else:
        print('Result: We fail to reject the null hypothesis; genre and country may be independent.\n')
else:
    print("Skipping Chi-square test, no overlapping data for top genres/countries.\n")


# Correlation: imdb_rating vs netflix addition speed?
lag_corr_result = lag_correlation(titles)
if lag_corr_result:
    print(f"Spearman correlation (Release Year vs. Addition Lag): rho = {lag_corr_result['spearman_rho']:.3f}, p-value = {lag_corr_result['p_value']:.3f}\n")

print("--- Text Analysis ---")
# Sentiment analysis
titles_with_sentiment = add_sentiment(titles)
merged_sentiment_genre = titles_with_sentiment.merge(genres, on='show_id')
print('Average sentiment by genre (Top 10):')
print(merged_sentiment_genre.groupby('genre')['sentiment'].mean().sort_values(ascending=False).head(10))

# TF-IDF Keywords
print('\nTop keywords from descriptions (TF-IDF):')
top_terms = tfidf_top_terms(merged_sentiment_genre, group_col='genre', top_k=5)
print(top_terms.head(15))

## 6. Deliverable: Full Visualization Portfolio (30+ Charts)

The analysis above shows the *key* strategic charts. As required by the problem statement, the following code cell will generate the **full portfolio of 30-40 visualizations** (using your `visualization_functions.py` script) and save them to the `visualizations/` directory. 

This portfolio includes:
* **Overview Charts**: KPIs, Composition Pie.
* **Temporal Analysis**: Growth over time, Stacked Area (Movie vs TV), Monthly Heatmap, Lag Distribution.
* **Geographic Insights**: Top Countries Bar, Choropleth Map.
* **Genre Intelligence**: Treemap, Genre Evolution Streamgraph, Cross-Genre Heatmap.
* **Rating & Audience**: Rating Distribution, Rating by Genre Heatmap.
* **Duration & Format**: Movie Duration Histogram, TV Season Histogram, Duration by Genre Box Plot.
* **Advanced Analysis**: Sentiment by Genre, Word Clouds.

In [ ]:
# This cell runs the full visualization generation script.
# All 30+ charts will be saved to the '/visualizations' directory.

print('Generating full visualization portfolio...')

# Importing the function from your analytics code
try:
    from analytics_code.visualization_functions import generate_visualization_portfolio
    generate_visualization_portfolio()
    print(f'Successfully generated and saved all visualizations to {VIZ_DIR}')
except ImportError:
    print("Could not import 'generate_visualization_portfolio'. Please ensure analytics_code is in the Python path.")
except Exception as e:
    print(f"An error occurred during visualization generation: {e}")

## 7. Final Strategic Recommendations

Based on the complete analysis, here are the top actionable recommendations:

1.  **Invest in African & Southeast Asian Content Hubs**: The data shows these markets are vastly underserved. Earmark a strategic fund (e.g., $100M) to develop and acquire content from emerging markets like Nigeria, South Africa, and Indonesia to capture a first-mover advantage[cite: 111, 122, 123].

2.  **Rebalance the Ratings Portfolio**: The catalog is heavily skewed towards mature, dramatic content (TV-MA Dramas)[cite: 113, 118]. Launch a dedicated initiative to acquire and produce high-quality, premium content for the underserved **"Family" (PG/TV-PG)** and **"Teen" (TV-14)** audiences.

3.  **Acquire High-Performing Niche Genres**: Analysis shows a clear viewer appetite for high-quality **Documentaries** and **Stand-Up Comedy**, but these genres are underrepresented in volume. Actively pursue the acquisition of award-winning content in these categories to satisfy this engaged audience[cite: 113, 122].

4.  **Diversify the Creator Pool**: The creator network analysis reveals a concentration among a small group of established talent[cite: 115]. Implement a **"New Voices" program** to actively fund and mentor first-time directors and writers from the identified underrepresented regions[cite: 115, 122, 123].